# 教師棋譜からのパターン評価学習

`training_records/` に世界1位AI同士の自己対戦棋譜 `.txt` を置いて実行します。

想定する棋譜形式:

- 1行 = 1局
- 手は `a1` から `h8` の2文字表記
- `f5d6c3...` のような連結形式でも、`f5 d6 c3 ...` の空白区切りでもOK

TensorFlow、numpy、外部評価器は使わず、標準ライブラリだけで `PATTERN_TABLES` を作ります。


In [ ]:
import pathlib
import random

RECORD_DIR = pathlib.Path("training_records")

EMPTY, BLACK, WHITE = 0, 1, 2
DIRECTIONS = ((-1,-1),(-1,0),(-1,1),(0,-1),(0,1),(1,-1),(1,0),(1,1))

# 参考コードと同じ3種類。index = row * 8 + col。
diagonal8_idx = [
    [0, 9, 18, 27, 36, 45, 54, 63],
    [7, 14, 21, 28, 35, 42, 49, 56],
]
diagonal8_idx += [list(reversed(pattern)) for pattern in diagonal8_idx]

edge_2x_idx = [
    [9, 0, 1, 2, 3, 4, 5, 6, 7, 14],
    [9, 0, 8, 16, 24, 32, 40, 48, 56, 49],
    [49, 56, 57, 58, 59, 60, 61, 62, 63, 54],
    [54, 63, 55, 47, 39, 31, 23, 15, 7, 14],
]
edge_2x_idx += [list(reversed(pattern)) for pattern in edge_2x_idx]

triangle_idx = [
    [0, 1, 2, 3, 8, 9, 10, 16, 17, 24],
    [0, 8, 16, 24, 1, 9, 17, 2, 10, 3],
    [7, 6, 5, 4, 15, 14, 13, 23, 22, 31],
    [7, 15, 23, 31, 6, 14, 22, 5, 13, 4],
    [63, 62, 61, 60, 55, 54, 53, 47, 46, 39],
    [63, 55, 47, 39, 62, 54, 46, 61, 53, 60],
    [56, 57, 58, 59, 48, 49, 50, 40, 41, 32],
    [56, 48, 40, 32, 57, 49, 41, 58, 50, 59],
]

PATTERN_INDEXES = {
    "diagonal8": diagonal8_idx,
    "edge2X": edge_2x_idx,
    "triangle": triangle_idx,
}

pattern_tables = {name: {} for name in PATTERN_INDEXES}
pattern_counts = {name: {} for name in PATTERN_INDEXES}

print("record dir:", RECORD_DIR)
print("patterns:", {name: len(patterns) for name, patterns in PATTERN_INDEXES.items()})

In [ ]:
def opponent(color):
    return WHITE if color == BLACK else BLACK

def initial_board():
    board = [[EMPTY for _ in range(8)] for _ in range(8)]
    board[3][3], board[4][4] = WHITE, WHITE
    board[3][4], board[4][3] = BLACK, BLACK
    return board

def flatten_board(board):
    return [cell for row in board for cell in row]

def flips_for(board, row, col, color):
    if board[row][col] != EMPTY:
        return []
    other = opponent(color)
    flips = []
    for dr, dc in DIRECTIONS:
        r, c = row + dr, col + dc
        line = []
        while 0 <= r < 8 and 0 <= c < 8 and board[r][c] == other:
            line.append((r, c))
            r += dr
            c += dc
        if line and 0 <= r < 8 and 0 <= c < 8 and board[r][c] == color:
            flips.extend(line)
    return flips

def legal_moves(board, color):
    return [(r, c) for r in range(8) for c in range(8) if flips_for(board, r, c, color)]

def apply_move(board, move, color):
    r, c = move
    next_board = [row[:] for row in board]
    next_board[r][c] = color
    for fr, fc in flips_for(board, r, c, color):
        next_board[fr][fc] = color
    return next_board

def final_result(board):
    black = sum(cell == BLACK for row in board for cell in row)
    white = sum(cell == WHITE for row in board for cell in row)
    vacant = 64 - black - white
    diff = black - white
    if diff > 0:
        diff += vacant
    elif diff < 0:
        diff -= vacant
    return diff / 64.0


In [ ]:
def parse_record_line(line):
    line = line.strip().lower()
    if not line or line.startswith("#"):
        return []
    for ch in ",;":
        line = line.replace(ch, " ")
    parts = line.split()
    if len(parts) == 1:
        text = parts[0]
        if len(text) % 2 != 0:
            raise ValueError(f"odd-length record: {line[:40]}")
        parts = [text[i:i+2] for i in range(0, len(text), 2)]
    moves = []
    for token in parts:
        if len(token) != 2 or token[0] < "a" or token[0] > "h" or token[1] < "1" or token[1] > "8":
            raise ValueError(f"bad move token: {token}")
        row = int(token[1]) - 1
        col = ord(token[0]) - ord("a")
        moves.append((row, col))
    return moves

def load_records(record_dir=RECORD_DIR, max_games=None):
    paths = sorted(record_dir.glob("*.txt"))
    records = []
    for path in paths:
        with path.open("r", encoding="utf-8") as file:
            for line_no, line in enumerate(file, 1):
                moves = parse_record_line(line)
                if moves:
                    records.append((path.name, line_no, moves))
                    if max_games is not None and len(records) >= max_games:
                        return records
    return records

def replay_record(moves, source="record"):
    board = initial_board()
    color = BLACK
    history = []
    for move_no, move in enumerate(moves, 1):
        if not legal_moves(board, color):
            color = opponent(color)
        if move not in legal_moves(board, color):
            raise ValueError(f"illegal move {move} at {source}, move {move_no}, color {color}")
        history.append((flatten_board(board), color))
        board = apply_move(board, move, color)
        color = opponent(color)
    return history, final_result(board)

records = load_records()
print("loaded games:", len(records))
if not records:
    print("Put teacher .txt records into training_records/ and run this cell again.")


In [ ]:
def pattern_key(flat_board, pattern):
    chars = []
    for idx in pattern:
        chars.append("1" if flat_board[idx] == BLACK else "0")
    for idx in pattern:
        chars.append("1" if flat_board[idx] == WHITE else "0")
    return "".join(chars)

def update_table(name, key, target, learning_rate):
    old = pattern_tables[name].get(key, 0.0)
    pattern_tables[name][key] = old + learning_rate * (target - old)
    pattern_counts[name][key] = pattern_counts[name].get(key, 0) + 1

def train_from_records(records, learning_rate=0.02, shuffle_games=True, seed=42):
    records = list(records)
    if shuffle_games:
        random.seed(seed)
        random.shuffle(records)

    n_positions = 0
    n_skipped = 0
    for game_no, (file_name, line_no, moves) in enumerate(records, 1):
        try:
            history, black_result = replay_record(moves, f"{file_name}:{line_no}")
        except ValueError as error:
            n_skipped += 1
            if n_skipped <= 5:
                print("skip:", error)
            continue

        for flat, color in history:
            target = black_result
            for name, patterns in PATTERN_INDEXES.items():
                for pattern in patterns:
                    update_table(name, pattern_key(flat, pattern), target, learning_rate)
            n_positions += 1

        if game_no % 1000 == 0 or game_no == len(records):
            print("games", game_no, "positions", n_positions, "skipped", n_skipped)

    return {"games": len(records), "positions": n_positions, "skipped": n_skipped}

summary = train_from_records(records, learning_rate=0.02, shuffle_games=True, seed=42)
summary

In [ ]:
def export_tables(min_count=3, digits=4):
    exported = {}
    for name, table in pattern_tables.items():
        exported[name] = {
            key: round(value, digits)
            for key, value in table.items()
            if pattern_counts[name].get(key, 0) >= min_count and abs(value) > 0.0001
        }
    return exported

PATTERN_TABLES = export_tables(min_count=3, digits=4)
text = "PATTERN_TABLES = " + repr(PATTERN_TABLES)
print("sizes:", {name: len(table) for name, table in PATTERN_TABLES.items()})
print(text[:4000])
if len(text) > 4000:
    print("... preview only. PATTERN_TABLES contains the full data.")


## 次にやること

`training_records/` に2万局分の `.txt` を置いた後、上から順番に実行してください。最後に出た `PATTERN_TABLES` を提出用 `MyPlayer` へ貼ります。
